<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/spark_sql_nested_json_repair_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spark SQL In-Class Assignment: Nested JSON Repair Lab

In this notebook you will work with **one nested JSON dataset** and complete missing or incorrect Spark SQL code.

Your goals:
- work with **arrays**, **structs**, and **maps**
- use **at least 3 `LATERAL VIEW` clauses** across the notebook
- repair incomplete SQL/PySpark code
- pass **3 checkpoints** during class

## Grading
- **Checkpoint 1** completed: **3 points**
- **Checkpoint 1 + 2** completed: **7 points**
- **Checkpoint 1 + 2 + 3** completed: **10 points**
- **Checkpoint 1 + 2 + 3 + Final challenge** completed: **15 points**

A checkpoint counts as completed when your final cell for that checkpoint runs and produces the expected result.



# 1. ⚡ Starting Spark

Before working with the data, we need to start a Spark session.

This is the entry point for working with Spark.

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Spark Nested JSON Repair Lab") \
    .getOrCreate()

spark

## 2. Environment Set Up

In [4]:
# @title
from pathlib import Path
import json

data_dir = Path('/content/spark_sql_nested_lab')
data_dir.mkdir(parents=True, exist_ok=True)
json_path = data_dir / 'festival_nested.json'

records = [
    {
        'event_id': 101,
        'city': 'Madrid',
        'meta': {'theme': 'AI', 'capacity': 120},
        'ticket_prices': {'standard': 35, 'vip': 60},
        'attendees': [
            {'name': 'Ana', 'country': 'ES', 'tags': ['student', 'python'], 'contacts': {'email': 'ana@example.com', 'telegram': '@ana'}},
            {'name': 'Leo', 'country': 'PT', 'tags': ['speaker', 'sql'], 'contacts': {'email': 'leo@example.com', 'telegram': '@leo'}}
        ],
        'sessions': [
            {
                'session_id': 'M1',
                'title': 'Intro to Spark',
                'duration_min': 45,
                'resources': {'slides': 'spark_intro.pdf', 'room': 'A1'},
                'speakers': [
                    {'name': 'Nora', 'social': {'linkedin': 'nora-li', 'x': '@nora'}},
                    {'name': 'Pablo', 'social': {'linkedin': 'pablo-li', 'x': '@pablo'}}
                ]
            },
            {
                'session_id': 'M2',
                'title': 'Nested JSON',
                'duration_min': 60,
                'resources': {'slides': 'nested_json.pdf', 'room': 'B1'},
                'speakers': [
                    {'name': 'Sara', 'social': {'linkedin': 'sara-li', 'x': '@sara'}}
                ]
            }
        ]
    },
    {
        'event_id': 102,
        'city': 'Barcelona',
        'meta': {'theme': 'Data', 'capacity': 90},
        'ticket_prices': {'standard': 30, 'vip': 55},
        'attendees': [
            {'name': 'Marta', 'country': 'ES', 'tags': ['analytics', 'student'], 'contacts': {'email': 'marta@example.com', 'telegram': '@marta'}},
            {'name': 'Omar', 'country': 'MA', 'tags': ['engineering'], 'contacts': {'email': 'omar@example.com', 'telegram': '@omar'}},
            {'name': 'Iris', 'country': 'ES', 'tags': ['sql', 'mentoring'], 'contacts': {'email': 'iris@example.com', 'telegram': '@iris'}}
        ],
        'sessions': [
            {
                'session_id': 'B1',
                'title': 'SQL Patterns',
                'duration_min': 50,
                'resources': {'slides': 'sql_patterns.pdf', 'room': 'C2'},
                'speakers': [
                    {'name': 'Leo', 'social': {'linkedin': 'leo-li', 'x': '@leo_data'}}
                ]
            },
            {
                'session_id': 'B2',
                'title': 'Maps and Structs',
                'duration_min': 40,
                'resources': {'slides': 'maps_structs.pdf', 'room': 'C3'},
                'speakers': [
                    {'name': 'Ana', 'social': {'linkedin': 'ana-li', 'x': '@ana_data'}},
                    {'name': 'Sara', 'social': {'linkedin': 'sara-li', 'x': '@sara_data'}}
                ]
            }
        ]
    },
    {
        'event_id': 103,
        'city': 'Valencia',
        'meta': {'theme': 'Cloud', 'capacity': 75},
        'ticket_prices': {'standard': 28, 'vip': 50},
        'attendees': [
            {'name': 'Hugo', 'country': 'ES', 'tags': ['cloud', 'student'], 'contacts': {'email': 'hugo@example.com', 'telegram': '@hugo'}},
            {'name': 'Aya', 'country': 'FR', 'tags': ['devops', 'speaker'], 'contacts': {'email': 'aya@example.com', 'telegram': '@aya'}}
        ],
        'sessions': [
            {
                'session_id': 'V1',
                'title': 'Distributed Systems',
                'duration_min': 55,
                'resources': {'slides': 'distributed.pdf', 'room': 'D1'},
                'speakers': [
                    {'name': 'Omar', 'social': {'linkedin': 'omar-li', 'x': '@omar_cloud'}}
                ]
            }
        ]
    }
]

with open(json_path, 'w', encoding='utf-8') as f:
    for row in records:
        f.write(json.dumps(row) + '\n')

print(json_path)

/content/spark_sql_nested_lab/festival_nested.json


In [20]:
# @title
import json

data = [
    {
        "event_id": 1,
        "event_name": "Tech Conference",
        "status": "active",
        "sessions": [
            {"session_id": 101, "topic": "AI", "room": "A"},
            {"session_id": 102, "topic": "Data", "room": "B"}
        ],
        "attendees": [
            {"name": "Alice", "role": "speaker", "preferences": {"level": "advanced", "track": "AI"}, "tags": ["ml", "ai"]},
            {"name": "Bob", "role": "guest", "preferences": {"level": "beginner", "track": "data"}, "tags": ["sql"]},
            {"name": "Carol", "role": "participant", "preferences": {"level": "advanced", "track": "AI"}, "tags": ["ai", "spark"]}
        ]
    },
    {
        "event_id": 2,
        "event_name": "Data Summit",
        "status": "inactive",
        "sessions": [
            {"session_id": 201, "topic": "SQL", "room": "C"}
        ],
        "attendees": [
            {"name": "David", "role": "participant", "preferences": {"level": "intermediate", "track": "data"}, "tags": ["sql", "analytics"]},
            {"name": "Eve", "role": "guest", "preferences": {"level": "advanced", "track": "security"}, "tags": ["security"]}
        ]
    },
    {
        "event_id": 3,
        "event_name": "AI Workshop",
        "status": "active",
        "sessions": [
            {"session_id": 301, "topic": "Deep Learning", "room": "A"},
            {"session_id": 302, "topic": "NLP", "room": "A"},
            {"session_id": 303, "topic": "Vision", "room": "B"}
        ],
        "attendees": [
            {"name": "Frank", "role": "speaker", "preferences": {"level": "advanced", "track": "AI"}, "tags": ["dl", "vision"]},
            {"name": "Grace", "role": "participant", "preferences": {"level": "advanced", "track": "AI"}, "tags": ["nlp", "ai"]},
            {"name": "Heidi", "role": "guest", "preferences": {"level": "beginner", "track": "AI"}, "tags": ["intro"]}
        ]
    },
    {
        "event_id": 4,
        "event_name": "Cloud Expo",
        "status": "active",
        "sessions": [
            {"session_id": 401, "topic": "AWS", "room": "D"},
            {"session_id": 402, "topic": "Azure", "room": "E"}
        ],
        "attendees": [
            {"name": "Ivan", "role": "participant", "preferences": {"level": "intermediate", "track": "cloud"}, "tags": ["aws"]},
            {"name": "Judy", "role": "participant", "preferences": {"level": "advanced", "track": "cloud"}, "tags": ["azure", "cloud"]}
        ]
    }
]

# Write as JSON lines (Spark-friendly format)
file_path = "events_nested.json"

with open(file_path, "w") as f:
    for row in data:
        f.write(json.dumps(row) + "\n")

print(f"File written to {file_path}")

File written to events_nested.json


## 3. Load the JSON

This cell is intentionally incomplete. Fix it.

In [38]:
# TODO: repair this cell
# Goal: read the JSON file festival_nested.json which is in the file location: /content/spark_sql_nested_lab
df = spark.read.__________

# Show a preview
df.______________

AttributeError: 'DataFrameReader' object has no attribute '__________'

## 4. Inspect the schema


In [ ]:
# TODO: print the schema
# write your code below


## 5. Create a temp view

You will use SQL for the checkpoints. So register our DataFrame as a view like called **festival**

In [8]:
# TODO: create or replace a temp view named festival
df.________________________

## Checkpoint 1 — First `LATERAL VIEW`

Repair the SQL so it returns **one row per attendee** with these columns:
- `event_id`
- `city`
- `attendee_name`
- `attendee_country`

Expected number of rows: **7**

In [9]:
# TODO: repair the SQL
q1 = spark.sql("""
SELECT
    event_id,
    city,
    ______________ AS attendee_name,
    ______________ AS attendee_country
FROM festival
LATERAL VIEW explode(__________) t AS attendee
ORDER BY event_id, attendee_name
""")

q1.show()

+--------+---------+-------------+----------------+
|event_id|     city|attendee_name|attendee_country|
+--------+---------+-------------+----------------+
|     101|   Madrid|          Ana|              ES|
|     101|   Madrid|          Leo|              PT|
|     102|Barcelona|         Iris|              ES|
|     102|Barcelona|        Marta|              ES|
|     102|Barcelona|         Omar|              MA|
|     103| Valencia|          Aya|              FR|
|     103| Valencia|         Hugo|              ES|
+--------+---------+-------------+----------------+



In [10]:
# Checkpoint 1 validation
assert q1.count() == 7, 'Checkpoint 1: expected 7 attendee rows'
print('Checkpoint 1 complete')

Checkpoint 1 complete


## 6. Access struct and map fields

Now work with direct nested access.

Return:
- `event_id`
- `theme` from the struct `meta`
- `vip_price` from the map `ticket_prices`

Expected rows: **3**

In [11]:
# TODO: complete the SQL
q_struct_map = spark.sql("""
SELECT
    event_id,
    ________ AS theme,
    ticket_prices['________'] AS vip_price
FROM festival
ORDER BY event_id
""")

q_struct_map.show()

+--------+-----+---------+
|event_id|theme|vip_price|
+--------+-----+---------+
|     101|   AI|       60|
|     102| Data|       55|
|     103|Cloud|       50|
+--------+-----+---------+



## Checkpoint 2 — Two `LATERAL VIEW` clauses

Now flatten sessions and then speakers.

Return:
- `city`
- `session_id`
- `session_title`
- `speaker_name`
- `room`

Requirements:
- use **2 `LATERAL VIEW` clauses** in the same query
- extract `room` from the `resources` map

Expected number of rows: **7**

In [14]:
# TODO: repair the SQL
q2 = spark.sql("""
SELECT
    ______________
FROM festival
______________
______________
ORDER BY city, session_id, speaker_name
""")

q2.show(truncate=False)

+---------+----------+-------------------+------------+----+
|city     |session_id|session_title      |speaker_name|room|
+---------+----------+-------------------+------------+----+
|Barcelona|B1        |SQL Patterns       |Leo         |C2  |
|Barcelona|B2        |Maps and Structs   |Ana         |C3  |
|Barcelona|B2        |Maps and Structs   |Sara        |C3  |
|Madrid   |M1        |Intro to Spark     |Nora        |A1  |
|Madrid   |M1        |Intro to Spark     |Pablo       |A1  |
|Madrid   |M2        |Nested JSON        |Sara        |B1  |
|Valencia |V1        |Distributed Systems|Omar        |D1  |
+---------+----------+-------------------+------------+----+



In [15]:
# Checkpoint 2 validation
assert q2.count() == 7, 'Checkpoint 2: expected 7 speaker-session rows'
print('Checkpoint 2 complete')

Checkpoint 2 complete


## 7. PySpark repair cell

This time do not use SQL. Repair the DataFrame code so it returns one row per attendee tag.

Columns:
- `city`
- `attendee_name`
- `tag`

Hint: you need **two explodes** because tags are inside each attendee.

In [16]:
# TODO: repair the PySpark code
from pyspark.sql.functions import explode, col

attendee_tags = (
    df
    .select(
        'city',
        explode('___________').alias('attendee')
    )
    .select(
        'city',
        col('attendee._______').alias('attendee_name'),
        explode(col('attendee._______')).alias('tag')
    )
)

attendee_tags.show()

+---------+-------------+-----------+
|     city|attendee_name|        tag|
+---------+-------------+-----------+
|   Madrid|          Ana|    student|
|   Madrid|          Ana|     python|
|   Madrid|          Leo|    speaker|
|   Madrid|          Leo|        sql|
|Barcelona|        Marta|  analytics|
|Barcelona|        Marta|    student|
|Barcelona|         Omar|engineering|
|Barcelona|         Iris|        sql|
|Barcelona|         Iris|  mentoring|
| Valencia|         Hugo|      cloud|
| Valencia|         Hugo|    student|
| Valencia|          Aya|     devops|
| Valencia|          Aya|    speaker|
+---------+-------------+-----------+



## Checkpoint 3 — Three `LATERAL VIEW` clauses

Final task: build a query with **3 `LATERAL VIEW` clauses**.

Return one row per combination of:
- event
- attendee
- attendee tag
- session

Columns:
- `city`
- `attendee_name`
- `tag`
- `session_id`
- `theme`

Conditions:
- only keep rows where `tag` is either `'student'` or `'sql'`
- order by `city`, `attendee_name`, `session_id`, `tag`

Requirements:
- 1st `LATERAL VIEW`: attendees
- 2nd `LATERAL VIEW`: attendee tags
- 3rd `LATERAL VIEW`: sessions

Expected number of rows: **9**

In [17]:
# TODO: complete the final SQL
q3 = spark.sql("""


""")

q3.show(truncate=False)

+---------+-------------+-------+----------+-----+
|city     |attendee_name|tag    |session_id|theme|
+---------+-------------+-------+----------+-----+
|Barcelona|Iris         |sql    |B1        |Data |
|Barcelona|Iris         |sql    |B2        |Data |
|Barcelona|Marta        |student|B1        |Data |
|Barcelona|Marta        |student|B2        |Data |
|Madrid   |Ana          |student|M1        |AI   |
|Madrid   |Ana          |student|M2        |AI   |
|Madrid   |Leo          |sql    |M1        |AI   |
|Madrid   |Leo          |sql    |M2        |AI   |
|Valencia |Hugo         |student|V1        |Cloud|
+---------+-------------+-------+----------+-----+



In [18]:
# Checkpoint 3 validation
assert q3.count() == 9, 'Checkpoint 3: expected 9 final rows'
print('Checkpoint 3 complete')

Checkpoint 3 complete


## 🔍 Filtering nested data without `LATERAL VIEW`

So far, you have used `LATERAL VIEW` and `explode()` to work with nested data.

However, in many cases, we **don’t want to explode the data**, we just want to filter rows based on conditions inside arrays.

Spark allows us to do this using functions like:

* `filter()`
* `exists()`
* `size()`


## How do we work with arrays without `explode()`?

Let’s start with a simple idea.

Suppose you have a column called `tags`, and it contains an array like this:

```sql
['ai', 'ml', 'data']
```

Now imagine you want to **remove `'ml'` from that array**.

In Spark SQL, you can write:

```sql
filter(tags, t -> t != 'ml')
```

### How to read this

Read it like this:

> “Go through the `tags` array, one element at a time (call it `t`), and keep it if it is not `'ml'`”

So:

* `tags` is the array
* `t` is **one element inside that array**
* `t -> ...` defines the rule for each element

Result:

```sql
['ai', 'data']
```

### What is `t`?

* `t` is just a temporary name
* it represents **one value from the array at a time**
* you can use another name (`x`, `a`, etc.)


### When the array has more structure

Now let’s connect this to our dataset.

The `attendees` column is also an array but each element is more complex.

Each attendee has fields like:

* `name`
* `role`
* `tags`

So we can write:

```sql
filter(attendees, a -> a.role != 'guest')
```

👉 Read it as:

> “Go through each attendee (`a`), and keep them if their role is not `'guest'`”

---

### The `array_contains()` function

Sometimes we need to check if an array contains a value.

```sql
array_contains(array, value)
```

Example:

```sql
array_contains(['ai', 'ml', 'data'], 'ai')
```

👉 returns `true`

### Using it with nested data

Each attendee has their own `tags` array.

So we can write:

```sql
array_contains(a.tags, 'ai')
```

👉 “Does this attendee have the tag `'ai'`?”


---

## 🚀 Now you try

In the next steps, you will apply this idea to the `attendees` array.


## Step 1 — Filter attendees inside the array

👉 Goal: Keep only attendees who are **not guests**

Let's read the data first:

In [62]:
events_df = spark.read.json("events_nested.json")
events_df.printSchema()
events_df.createOrReplaceTempView("practice_events")

root
 |-- attendees: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- preferences: struct (nullable = true)
 |    |    |    |-- level: string (nullable = true)
 |    |    |    |-- track: string (nullable = true)
 |    |    |-- role: string (nullable = true)
 |    |    |-- tags: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |-- event_id: long (nullable = true)
 |-- event_name: string (nullable = true)
 |-- sessions: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- room: string (nullable = true)
 |    |    |-- session_id: long (nullable = true)
 |    |    |-- topic: string (nullable = true)
 |-- status: string (nullable = true)




Complete the query:

In [58]:
spark.sql("""
SELECT
    event_id,
    attendees,
    filter(attendees, a -> ______ != '_____') AS filtered_attendees
FROM practice_events
""").show(truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------
 event_id           | 1                                                                                                                                          
 attendees          | [{Alice, {advanced, AI}, speaker, [ml, ai]}, {Bob, {beginner, data}, guest, [sql]}, {Carol, {advanced, AI}, participant, [ai, spark]}]     
 filtered_attendees | [{Alice, {advanced, AI}, speaker, [ml, ai]}, {Carol, {advanced, AI}, participant, [ai, spark]}]                                            
-RECORD 1--------------------------------------------------------------------------------------------------------------------------------------------------------
 event_id           | 2                                                                                                                                          
 attendees          | [{Davi

Now make it more specific:

👉 Keep only events where at least one attendee:

* is **not a guest**
* AND has `"ai"` in their `tags`

Complete the query:

Note: use array_contains function

In [64]:
spark.sql("""
SELECT event_id, event_name
FROM events_nested
WHERE size(
    filter(attendees, a ->
        ________ AND ____________
    )
) > 0
""").show(truncate=False, vertical=True)

-RECORD 0---------------------
 event_id   | 1               
 event_name | Tech Conference 
-RECORD 1---------------------
 event_id   | 3               
 event_name | AI Workshop     



## 🧩 Final Challenge — Why didn’t anything change?

In this last part, you will debug a small PySpark pipeline.

A classmate tried to clean the dataset before running a SQL query, but something went wrong. The transformations were not applied correctly, so the temporary view was not updated as expected.

Because of that, the SQL query below does not work correctly.

In the next cell, just run the code. It will load the dataset into a DataFrame so you can inspect its structure before debugging the problem.


In [30]:

events_df = spark.read.json("events_nested.json")
events_df.printSchema()
events_df.show()

root
 |-- attendees: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- preferences: struct (nullable = true)
 |    |    |    |-- level: string (nullable = true)
 |    |    |    |-- track: string (nullable = true)
 |    |    |-- role: string (nullable = true)
 |    |    |-- tags: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |-- event_id: long (nullable = true)
 |-- event_name: string (nullable = true)
 |-- sessions: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- room: string (nullable = true)
 |    |    |-- session_id: long (nullable = true)
 |    |    |-- topic: string (nullable = true)
 |-- status: string (nullable = true)

+--------------------+--------+---------------+--------------------+--------+
|           attendees|event_id|     event_name|            sessions|  status|
+--------------------+--------+---------------+---

The next cell contains the bug.

Your task is to fix the PySpark code so that the SQL query below can run correctly.

Right now, the transformations are not being saved properly, and the temporary view is being created from the wrong DataFrame. As a result, the view does not include the expected cleaned data or the new session_count column.

### Your goal

Fix the PySpark code so that:

* only **active events** are kept
* attendees with role **`"guest"`** are removed
* a new column called **`session_count`** is added
* the temporary view **`events_nested`** is recreated from the final DataFrame
* the SQL query below runs successfully

### Keep these ideas in mind

* Spark DataFrames **cannot be changed in place**
* transformations like `.withColumn()` return a **new DataFrame**
* Spark SQL reads from a **temporary view**, not directly from your Python variable
* if the DataFrame changes, the temporary view must be **created again**

In [33]:
from pyspark.sql.functions import size, expr

events_step1 = events_df.filter("status = 'active'")

events_step1.withColumn(
    "attendees",
    expr("filter(attendees, x -> x.role != 'guest')")
)

events_step1.withColumn(
    "session_count",
    size("sessions")
)

events_df.createOrReplaceTempView("events_nested")

The query below should work after you fix the previous cell.

At the moment, it may fail or return incorrect results because the pipeline is not properly applied.

Once you fix the code, the query should run successfully and return **6 rows** like this:

| event_id | session_count | name  |
| -------- | ------------- | ----- |
| 1        | 2             | Alice |
| 1        | 2             | Carol |
| 3        | 3             | Frank |
| 3        | 3             | Grace |
| 4        | 2             | Ivan  |
| 4        | 2             | Judy  |

In [37]:
spark.sql("""
SELECT
    event_id,
    session_count,
    attendee.name
FROM events_nested
LATERAL VIEW explode(attendees) t AS attendee
WHERE session_count >= 2
ORDER BY event_id, attendee.name
""").show()

+--------+-------------+-----+
|event_id|session_count| name|
+--------+-------------+-----+
|       1|            2|Alice|
|       1|            2|Carol|
|       3|            3|Frank|
|       3|            3|Grace|
|       4|            2| Ivan|
|       4|            2| Judy|
+--------+-------------+-----+

